In [2]:
import numpy as np
import OpenVisus as ov
import matplotlib.pyplot as plt
import os
from pathlib import Path
import pandas as pd
import pyvista as pv
from tqdm import tqdm

In [ ]:
# Params
num_frames = 500 # for 500 frames, notable hurricane pattern is 0-100, 350-480
face = 2
quality = -6
frames = np.linspace(0, 10268, num_frames, dtype=int).tolist()
variables = ['u','v','w']
output_dir = Path(f"velocity_vtk({quality})")
pvd_path = output_dir / "velocity_series.pvd"

In [13]:
# initialize databases
output_dir.mkdir(exist_ok=True)
times = np.linspace(0, 1, num_frames)

dbs = []
urls = []
for variable in variables:
    geos_face_loc=f"https://nsdf-climate3-origin.nationalresearchplatform.org:50098/nasa/nsdf/climate3/dyamond/GEOS/GEOS_{variable.upper()}/{variable.lower()}_face_{face}_depth_52_time_0_10269.idx"
    urls.append(geos_face_loc)
    dbs.append(ov.LoadDataset(geos_face_loc))
# Create coordinate indices
data = dbs[0].read(
    time=0,
    quality=quality
)
Z, Y, X = data.shape
grid = pv.ImageData()
grid.dimensions = (X, Y, Z)
grid.origin = (0, 0, 0)
grid.spacing = (1, 1, 1)
print(data.shape)

(52, 180, 180)


In [14]:
for i, t in enumerate(tqdm(frames, desc="Generating VTK files")):
    # if (i < 302):
    #     continue
    u = dbs[0].read(time=t, quality=quality)
    v = dbs[1].read(time=t, quality=quality)
    w = dbs[2].read(time=t, quality=quality)
    vectors = np.stack((u.flatten(), v.flatten(), w.flatten()), axis=1)
    # Flatten everything
    grid["wind_velocity"] = vectors.reshape(-1, 3)
    filename = output_dir / f"frame_{i:04d}.vti"
    grid.save(filename)

Generating VTK files: 100%|██████████| 500/500 [51:23<00:00,  6.17s/it]  


In [15]:
files = []
for i, t in enumerate(frames):
    filename = output_dir / f"frame_{i:04d}.vti"
    files.append((i, filename.name))


In [16]:

with open(pvd_path, "w") as f:
    f.write('<?xml version="1.0"?>\n')
    f.write('<VTKFile type="Collection" version="0.1">\n')
    f.write('  <Collection>\n')
    
    for t, fname in tqdm(files, desc="Writing PVD"):
        f.write(f'    <DataSet timestep="{t}" file="{fname}"/>\n')
    
    f.write('  </Collection>\n')
    f.write('</VTKFile>\n')

Writing PVD: 100%|██████████| 500/500 [00:00<00:00, 250137.40it/s]
